# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by Marten Postma and Filip Ilievski and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

In [2]:
import os
from nltk.corpus.reader import ConllCorpusReader

# 注意:解压后多了一层 CONLL2003 文件夹,所以要写两次
DATA_DIR = os.path.join('CONLL2003', 'CONLL2003')

# ---- Training data ----
train = ConllCorpusReader(DATA_DIR, 'train.txt', ['words', 'pos', 'ignore', 'chunk'])

training_features = []
training_gold_labels = []

for token, pos, ne_label in train.iob_words():
    a_dict = {
        'words': token,
        'pos': pos
    }
    training_features.append(a_dict)
    training_gold_labels.append(ne_label)

print("Training instances:", len(training_features))
print("First 3 features:", training_features[:3])
print("First 3 labels:", training_gold_labels[:3])

/opt/anaconda3/envs/CSE412A/lib/python3.10/site-packages/nltk/data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path /Users/yuxitian/Desktop/荷兰留学/TextMining for AI/ba-text-mining-master/lab_sessions/lab4/CONLL2003/CONLL2003/train.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


Training instances: 203621
First 3 features: [{'words': 'EU', 'pos': 'NNP'}, {'words': 'rejects', 'pos': 'VBZ'}, {'words': 'German', 'pos': 'JJ'}]
First 3 labels: ['B-ORG', 'O', 'B-MISC']


In [3]:
# ---- Test data ----
test = ConllCorpusReader(DATA_DIR, 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []

for token, pos, ne_label in test.iob_words():
    a_dict = {
        'words': token,
        'pos': pos
    }
    test_features.append(a_dict)
    test_gold_labels.append(ne_label)

print("Test instances:", len(test_features))
print("First 3 features:", test_features[:3])
print("First 3 labels:", test_gold_labels[:3])

Test instances: 46435
First 3 features: [{'words': 'SOCCER', 'pos': 'NN'}, {'words': '-', 'pos': ':'}, {'words': 'JAPAN', 'pos': 'NNP'}]
First 3 labels: ['O', 'O', 'B-LOC']


/opt/anaconda3/envs/CSE412A/lib/python3.10/site-packages/nltk/data.py:388: RuntimeWarning: Security Warning [pathsec.open]: Path /Users/yuxitian/Desktop/荷兰留学/TextMining for AI/ba-text-mining-master/lab_sessions/lab4/CONLL2003/CONLL2003/test.txt allowed via CWD.
  stream = _secure_open(self._path, "rb")


**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [4]:
from collections import Counter 

my_list=[1,2,1,3,2,5]
Counter(my_list)

print("Number of training instances:", len(training_features))
print("Number of test instances:    ", len(test_features))
print()

# 2. NERC 标签的频率分布
train_label_counts = Counter(training_gold_labels)
test_label_counts  = Counter(test_gold_labels)

print("Training label distribution:")
for label, count in sorted(train_label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:10s} {count:>7d}  ({count/len(training_gold_labels)*100:5.2f}%)")

print("\nTest label distribution:")
for label, count in sorted(test_label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:10s} {count:>7d}  ({count/len(test_gold_labels)*100:5.2f}%)")

Number of training instances: 203621
Number of test instances:     46435

Training label distribution:
  O           169578  (83.28%)
  B-LOC         7140  ( 3.51%)
  B-PER         6600  ( 3.24%)
  B-ORG         6321  ( 3.10%)
  I-PER         4528  ( 2.22%)
  I-ORG         3704  ( 1.82%)
  B-MISC        3438  ( 1.69%)
  I-LOC         1157  ( 0.57%)
  I-MISC        1155  ( 0.57%)

Test label distribution:
  O            38323  (82.53%)
  B-LOC         1668  ( 3.59%)
  B-ORG         1661  ( 3.58%)
  B-PER         1617  ( 3.48%)
  I-PER         1156  ( 2.49%)
  I-ORG          835  ( 1.80%)
  B-MISC         702  ( 1.51%)
  I-LOC          257  ( 0.55%)
  I-MISC         216  ( 0.47%)


### Discussion: Descriptive statistics of the CoNLL-2003 train and test data

#### 1. Number of instances
- **Training set**: 203,621 token-level instances
- **Test set**: 46,435 token-level instances
- The test set is roughly **23%** the size of the training set, which is a typical train/test ratio.

#### 2. Frequency distribution of NERC labels

| Label   | Train count | Train %  | Test count | Test %   |
|---------|------------:|---------:|-----------:|---------:|
| O       |     169,578 | 83.28 %  |     38,323 | 82.53 %  |
| B-LOC   |       7,140 |  3.51 %  |      1,668 |  3.59 %  |
| B-PER   |       6,600 |  3.24 %  |      1,617 |  3.48 %  |
| B-ORG   |       6,321 |  3.10 %  |      1,661 |  3.58 %  |
| I-PER   |       4,528 |  2.22 %  |      1,156 |  2.49 %  |
| I-ORG   |       3,704 |  1.82 %  |        835 |  1.80 %  |
| B-MISC  |       3,438 |  1.69 %  |        702 |  1.51 %  |
| I-LOC   |       1,157 |  0.57 %  |        257 |  0.55 %  |
| I-MISC  |       1,155 |  0.57 %  |        216 |  0.47 %  |

#### 3. Are the data balanced? Do train and test differ?

**The data is highly imbalanced.** In both the training and test sets, the `O` label (non-entity tokens) dominates, accounting for **~83%** of all tokens. This is expected for token-level NERC, since most words in natural text are not part of a named entity. The remaining ~17% is split across 8 entity sub-labels (B-/I- × LOC/PER/ORG/MISC), each of which individually accounts for less than 4% of the data. The rarest labels — `I-LOC` and `I-MISC` — make up only around 0.5% each.

**Training and test distributions are very similar.** Comparing the two columns side by side, the percentage of each label is almost identical (e.g., `O` is 83.28% in train vs. 82.53% in test; `B-LOC` is 3.51% vs. 3.59%). This indicates that the train/test split is well stratified and drawn from the same underlying distribution, which is desirable for a fair evaluation. The only minor differences are that **`B-ORG` is slightly more frequent in test** (3.58% vs. 3.10%) and **`I-MISC` is slightly less frequent in test** (0.47% vs. 0.57%), but these differences are small and unlikely to have a major impact on evaluation.


**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [6]:
from sklearn.feature_extraction import DictVectorizer

In [7]:
vec = DictVectorizer()
combined_features = training_features + test_features
the_array = vec.fit_transform(combined_features)

print("Combined sparse matrix shape:", the_array.shape)

n_train = len(training_features)
train_array = the_array[:n_train]
test_array  = the_array[n_train:]

print("Train array shape:", train_array.shape)
print("Test  array shape:", test_array.shape)

#sanity check
assert train_array.shape[0] == len(training_features)
assert test_array.shape[0]  == len(test_features)
print("size matches.")

Combined sparse matrix shape: (250056, 27361)
Train array shape: (203621, 27361)
Test  array shape: (46435, 27361)
size matches.


**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [8]:
from sklearn import svm

In [9]:
lin_clf = svm.LinearSVC()

In [3]:
##### [ YOUR CODE SHOULD GO HERE ]
# lin_clf.fit( # your code here

**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [10]:
# your code here

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets (see Lab4a.1-NERC-introduction.ipynb)
path = '/Users/piek/Desktop/ONDERWIJS/data/nerc_datasets/kaggle/ner_v2.csv'
kaggle_dataset = pandas.read_csv(path, on_bad_lines='skip', encoding='unicode_escape')

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

## End of this notebook